# Day 10 — Deep Vision Architectures & ResNet

**Agri-AI Bootcamp** · Phase 3 of 5: *CNNs & Crop Disease* · Days 9–12

> Implement a lightweight ResNet block with an identity shortcut and plug it into the Day 9 classifier.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

In very deep nets, gradients shrink as they travel back and early layers stop learning. A residual block computes F(x) + x, giving the signal a direct shortcut. If extra layers aren’t useful, the block can learn F(x) ≈ 0 and pass x through.

### 🌾 Agriculture example

Residual blocks let the leaf classifier go deeper to separate look-alikes such as early and late blight without training stalling.

> **Watch out:** Shapes must match at the addition; use a 1×1 convolution on the shortcut when channels or stride change.

## 2. Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)

## 3. Data

In [ ]:
# ---- Leaf image helper -------------------------------------------------------
# Real course dataset: PlantVillage / Tomato leaf images on Kaggle (needs a
# kaggle.json API token).  So that this notebook runs anywhere, we fall back to
# a small SYNTHETIC leaf set: green patches = "healthy", green patches with
# brown lesions = "diseased".  Swap in the real folder when you have it.
import numpy as np, torch

def synth_leaves(n=400, size=64, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, 3, size, size), dtype=np.float32)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        leaf = np.stack([rng.uniform(.10, .25, (size, size)),
                         rng.uniform(.45, .70, (size, size)),
                         rng.uniform(.10, .25, (size, size))])
        diseased = i % 2
        if diseased:                      # add 3-6 brown lesions
            for _ in range(rng.integers(3, 7)):
                cx, cy = rng.integers(8, size - 8, 2)
                r = rng.integers(4, 9)
                yy, xx = np.ogrid[:size, :size]
                m = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
                leaf[0][m], leaf[1][m], leaf[2][m] = .45, .28, .10
        X[i] = leaf
        y[i] = diseased
    return torch.tensor(X), torch.tensor(y)

CLASSES = ["healthy", "diseased"]
X_img, y_img = synth_leaves()
print("images:", tuple(X_img.shape), "| labels:", torch.bincount(y_img).tolist())

perm = torch.randperm(len(y_img)); X_img, y_img = X_img[perm], y_img[perm]
cut = int(.8*len(y_img))
tr_dl = DataLoader(TensorDataset(X_img[:cut], y_img[:cut]), batch_size=32, shuffle=True)
te_dl = DataLoader(TensorDataset(X_img[cut:], y_img[cut:]), batch_size=64)

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
class Block(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.f=nn.Sequential(nn.Conv2d(c,c,3,padding=1),nn.BatchNorm2d(c),nn.ReLU(),
                          nn.Conv2d(c,c,3,padding=1),nn.BatchNorm2d(c))
    def forward(s,x): return torch.relu(s.f(x)+x)   # skip connection
```

## 5. Worked solution

In [ ]:
# ---- Residual block: output = relu(F(x) + x) --------------------------------
class Block(nn.Module):
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False),     nn.BatchNorm2d(c_out))
        # shapes must match at the +; 1x1 conv fixes channels/stride
        self.skip = (nn.Identity() if (stride == 1 and c_in == c_out) else
                     nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False),
                                   nn.BatchNorm2d(c_out)))
    def forward(self, x): return torch.relu(self.f(x) + self.skip(x))

b = Block(16, 16).to(dev)
x = torch.randn(2, 16, 32, 32, device=dev)
print("identity shortcut:", tuple(x.shape), "->", tuple(b(x).shape))
b2 = Block(16, 32, stride=2).to(dev)
print("1x1 shortcut     :", tuple(x.shape), "->", tuple(b2(x).shape))
try:
    bad = nn.Conv2d(16, 32, 3, padding=1).to(dev)(x) + x       # channels differ
except RuntimeError as e:
    print("\nwithout the 1x1 shortcut ->", str(e)[:95])

# ---- A small ResNet for leaves ----------------------------------------------
class LeafResNet(nn.Module):
    def __init__(self, n_cls=2, blocks=(16, 32, 64)):
        super().__init__()
        layers = [nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU()]
        c = 16
        for c_out in blocks:
            layers += [Block(c, c_out, stride=2), Block(c_out, c_out)]
            c = c_out
        self.body = nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(c, n_cls))
    def forward(self, x): return self.body(x)

def train(net, epochs=8, tag=""):
    net = net.to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3)
    lossf = nn.CrossEntropyLoss(); accs = []
    for ep in range(epochs):
        net.train()
        for xb, yb in tr_dl:
            opt.zero_grad(); lossf(net(xb.to(dev)), yb.to(dev)).backward(); opt.step()
        net.eval(); c = t = 0
        with torch.no_grad():
            for xb, yb in te_dl:
                c += (net(xb.to(dev)).argmax(1).cpu() == yb).sum().item(); t += len(yb)
        accs.append(c/t)
    print(f"{tag:<28} final test acc {accs[-1]:.3f} | params {sum(p.numel() for p in net.parameters()):,}")
    return accs

# plain deep stack (no shortcuts) vs the same depth WITH shortcuts
plain = nn.Sequential(*([nn.Conv2d(3,16,3,1,1), nn.BatchNorm2d(16), nn.ReLU()] +
                        sum([[nn.Conv2d(16,16,3,1,1), nn.BatchNorm2d(16), nn.ReLU()]
                             for _ in range(10)], []) +
                        [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 2)]))
a_plain = train(plain, tag="plain 11-conv stack")
a_res   = train(LeafResNet(), tag="residual net (same depth)")

plt.plot(a_plain, label="plain"); plt.plot(a_res, label="residual")
plt.xlabel("epoch"); plt.ylabel("test accuracy"); plt.legend()
plt.title("shortcuts keep the gradient flowing"); plt.show()

# ---- Gradient at the FIRST layer: the vanishing-gradient story --------------
for name, net in [("plain", plain.to(dev)), ("residual", LeafResNet().to(dev))]:
    net.zero_grad()
    nn.CrossEntropyLoss()(net(X_img[:16].to(dev)), y_img[:16].to(dev)).backward()
    g = next(p.grad.abs().mean().item() for p in net.parameters() if p.grad is not None)
    print(f"{name:<9} mean |grad| at first conv: {g:.3e}")

## 6. Your turn

1. Deepen `blocks` to (16, 32, 64, 128) and see whether accuracy or just training time grows.
2. Delete `+ self.skip(x)` from forward and re-train — reproduce the stall yourself.
3. Compare your LeafResNet parameter count with `torchvision.models.resnet18`.

In [ ]:
# TODO 1: deeper
# train(LeafResNet(blocks=(16,32,64,128)), tag="deeper residual")

# TODO 2: break the skip connection on purpose
class NoSkip(Block):
    def forward(self, x): return torch.relu(self.f(x))        # shortcut removed
# plug NoSkip into LeafResNet and re-train

# TODO 3: parameter comparison
from torchvision import models
print("resnet18 params:", sum(p.numel() for p in models.resnet18().parameters()))
print("LeafResNet params:", sum(p.numel() for p in LeafResNet().parameters()))

## 7. Dataset and references

**Dataset:** [Kaggle: PlantVillage leaf images](https://www.kaggle.com/datasets/emmarex/plantdisease)

**References**

- [Deep Residual Learning (paper)](https://arxiv.org/abs/1512.03385)

**Course material:** He et al.: Deep Residual Learning

---
**Next:** Day 11 — Transfer Learning & Data Augmentation